# TradePose Market Profile Strategy Guide

這份 notebook 是給新使用者或 LLM 使用的端到端範例：先用 `BatchTester` 查詢 instrument，再建立一個極簡 `StrategyConfig`，透過 OHLCV 下載 Market Profile 指標，最後簡短帶到 trades 產生流程。

使用順序：
1. 安裝套件與設定 API key。
2. 查詢 XAUUSD instrument。
3. 建立極簡策略：`open > VAH` 進場、`open < POC` 出場。
4. 下載含 Market Profile 指標的 OHLCV。
5. 查看 Market Profile struct 與 `tpo_distribution` 內部值。
6. 用 `submit()` 產生 trades，後續可分析 MAE / MFE / PnL。


In [ ]:
# Colab / Jupyter dependency setup
# 本機 VS Code 使用 uv sync 後的 .venv kernel 時可略過。
!pip install tradepose-client --ignore-requires-python --quiet


In [ ]:
import os
from getpass import getpass

import polars as pl
from dotenv import load_dotenv

from tradepose_client import BatchTester, Freq, TradeDirection
from tradepose_client.batch import Period

# 本機使用 repo root 的 .env；Colab 也可以上傳或手動建立 .env。
load_dotenv()

API_KEY = os.getenv("TRADEPOSE_API_KEY") or getpass("TradePose API key: ")
SERVER_URL = os.getenv("TRADEPOSE_SERVER_URL", "https://api.tradepose.com")

tester = BatchTester(api_key=API_KEY, server_url=SERVER_URL, poll_interval=2.0)
print(f"Connected to {SERVER_URL}")


In [ ]:
# 下載/查詢 instrument metadata。
# 先用 symbol 模糊搜尋，確認實際可用的 instrument key 與 tick_size。
instruments = tester.list_instruments(symbol="XAUUSD", limit=20)

print(f"count={instruments.count}, total={instruments.total}")
for idx, inst in enumerate(instruments.instruments):
    print(idx, inst.key, inst)


## Strategy Code

這個策略只保留觀察指標需要的最小結構：

- Daily Market Profile
- Initial Balance Market Profile
- ATR
- `open > VAH` 進場、`open < POC` 出場

`volatility_indicator` 會用 ATR，後續 trades 分析可用它正規化 MAE / MFE，例如 `mae / ATR`。`volatility_level` 是市場 regime 分層，不一定要設定；本範例保留一個簡單 ATR quantile 版本，設定後 trades 會帶出相關 level 欄位，方便依低/中/高波動分組。


In [ ]:
"""Simple Market Profile Strategy

目標是展示如何下載 OHLCV 與 Market Profile 指標，而不是展示複雜交易邏輯。
進出場規則刻意簡化為：open > VAH 進場，open < POC 出場。
"""

from zoneinfo import ZoneInfo

import polars as pl
from pydantic import Field
from tradepose_client import BlueprintBuilder, Freq, IndicatorType, StrategyBuilder, TradeDirection, TrendType
from tradepose_models.indicators.market_profile import (
    create_daily_mode,
    create_intraday_mode,
    create_profile_shape_config,
)
from tradepose_models.strategy import StrategyConfig, StrategyParams, VolatilityLevelConfig


class SimpleMarketProfileParams(StrategyParams):
    """極簡 Market Profile 範例策略參數。"""

    instrument: str = Field(description="商品代碼，如 PEPPERSTONE:spot:XAUUSD")
    base_freq: Freq = Field(description="基準頻率")
    trade_direction: TradeDirection = Field(description="交易方向")
    tz: ZoneInfo = Field(default_factory=lambda: ZoneInfo("UTC"), description="策略時區")
    ib_start_hour: int = Field(default=9, ge=0, le=23, description="IB 起始小時")
    ib_end_hour: int = Field(default=11, ge=0, le=23, description="IB 結束小時")
    tick_size: float = Field(default=1.0, gt=0, description="Market Profile tick size")
    atr_freq: Freq = Field(default=Freq.HOUR_1, description="ATR 頻率")
    atr_period: int = Field(default=120, ge=1, description="ATR 週期")
    include_volatility_level: bool = Field(default=True, description="是否計算 ATR regime level")

    def create_strategy(self) -> StrategyConfig:
        builder = StrategyBuilder(params=self)

        daily_mp = builder.add_indicator(
            IndicatorType.MARKET_PROFILE,
            mode=create_daily_mode(self.ib_start_hour, 0),
            shape_config=create_profile_shape_config(
                pshape_concentration_threshold=0.5,
                bshape_valley_threshold=0.5,
            ),
            tick_size=self.tick_size,
            value_area_pct=0.70,
            freq=Freq.MIN_30,
            shift=0,
        )

        builder.add_indicator(
            IndicatorType.MARKET_PROFILE,
            mode=create_intraday_mode(self.ib_start_hour, 0, self.ib_end_hour, 0),
            shape_config=create_profile_shape_config(
                pshape_concentration_threshold=0.5,
                bshape_valley_threshold=0.5,
            ),
            tick_size=self.tick_size,
            value_area_pct=0.70,
            freq=Freq.MIN_30,
            shift=0,
        )

        atr = builder.add_indicator(
            IndicatorType.ATR,
            period=self.atr_period,
            freq=self.atr_freq,
            shift=1,
        )

        volatility_level = None
        if self.include_volatility_level:
            atr_daily = builder.add_indicator(IndicatorType.ATR, period=21, freq=Freq.DAY_1, shift=1)
            atr_daily_prev = builder.add_indicator(IndicatorType.ATR, period=21, freq=Freq.DAY_1, shift=2)
            atr_daily_q1 = builder.add_indicator(
                IndicatorType.ATR_QUANTILE,
                atr_column=atr_daily.display_name(),
                window=60,
                quantile=0.25,
                freq=Freq.DAY_1,
                shift=0,
            )
            atr_daily_q2 = builder.add_indicator(
                IndicatorType.ATR_QUANTILE,
                atr_column=atr_daily.display_name(),
                window=60,
                quantile=0.50,
                freq=Freq.DAY_1,
                shift=0,
            )
            atr_daily_q3 = builder.add_indicator(
                IndicatorType.ATR_QUANTILE,
                atr_column=atr_daily.display_name(),
                window=60,
                quantile=0.75,
                freq=Freq.DAY_1,
                shift=0,
            )
            volatility_level = VolatilityLevelConfig(
                source=atr_daily.col(),
                source_prev=atr_daily_prev.col(),
                q1=atr_daily_q1.col(),
                q2=atr_daily_q2.col(),
                q3=atr_daily_q3.col(),
            )

        vah = daily_mp.market_profile.vah.forward_fill()
        poc = daily_mp.market_profile.poc.forward_fill()
        entry_expr = pl.col("open") > vah
        exit_expr = pl.col("open") < poc

        base_bp = (
            BlueprintBuilder(
                name="mp_vah_poc",
                direction=self.trade_direction,
                trend_type=TrendType.REVERSAL,
            )
            .add_immediate_entry_trigger(conditions=[entry_expr])
            .add_immediate_exit_trigger(conditions=[exit_expr])
            .build()
        )

        builder.set_base_blueprint(base_bp)
        return builder.build(
            volatility_indicator=atr.col(),
            volatility_level=volatility_level,
            note="Minimal Market Profile example; entry open > VAH, exit open < POC.",
        )


## 建立 StrategyConfig

`StrategyConfig` 可以先理解成三個部分：

- `base_instrument` / `base_freq`：策略使用哪個商品與基準時間軸。
- `indicators`：需要計算的指標，本範例包含 Market Profile 與 ATR。indicator 內部也可以指定其他商品，例如策略交易 `PEPPERSTONE:spot:XAUUSD`，但某個 indicator 使用 `PEPPERSTONE:spot:NAS100`；後端會依 instrument / freq 自動載入並 join 到計算資料中。
- `base_blueprint`：極簡進出場規則，僅用來讓 `submit()` 能產生 trades。

`StrategyConfig` 的 instrument 請使用查詢結果 object 的 `inst.key`。例如可用 key 會長得像 `PEPPERSTONE:spot:XAUUSD` 或 `PEPPERSTONE:spot:NAS100`；選定後再把 key 放進策略參數。

`strategy.name` 是 SDK 用 zlib + base64url 壓縮出的 machine-readable reference。直接 `print(strategy.name)` 會看到 `sp:v1z:...`；用 `SimpleMarketProfileParams.decode_label()` 可以轉回可讀參數摘要。


In [ ]:
# 推薦從上一格 instruments.instruments 的 object 取得 key。
# 例如：INSTRUMENT = instruments.instruments[0].key
INSTRUMENT = "PEPPERSTONE:spot:XAUUSD"

strategy = SimpleMarketProfileParams(
    instrument=INSTRUMENT,
    base_freq=Freq.MIN_15,
    trade_direction=TradeDirection.LONG,
    tick_size=1.0,
    atr_freq=Freq.HOUR_1,
    atr_period=120,
    include_volatility_level=True,
).create_strategy()

print(strategy.name)
print(SimpleMarketProfileParams.decode_label(strategy.name))
print(strategy.base_instrument, strategy.base_freq)
print(f"indicators={len(strategy.indicators)}")
print(f"base_blueprint={strategy.base_blueprint.name}")


In [ ]:
# 簡短檢查 StrategyConfig JSON 結構。
strategy_json = strategy.model_dump(mode="json", exclude_none=True)
print(strategy_json.keys())
print("first indicator:")
strategy_json["indicators"][0]


## 取回 Market Profile 指標計算結果

`submit_ohlcv()` 會從 strategy 抽出所有 indicator，提交 on-demand OHLCV task，背景輪詢下載完成後可從 `ohlcv_result.data` 或 `ohlcv_result.df` 讀到基礎 OHLCV 加上指標欄位。

注意：`OHLCVPeriodResult` 不提供 `wait()` method，所以這裡提交後先短暫等待，再讀 `.data`。如果資料尚未下載完成，可以稍後重新讀 `ohlcv_result.data`。


In [ ]:
period = Period(start="2025-01-01", end="2027-02-01")

ohlcv_result = tester.submit_ohlcv(strategy=strategy, period=period, timeout=300)

import time
time.sleep(5)

ohlcv_df = ohlcv_result.data
ohlcv_df


In [ ]:
# Market Profile 指標欄位是 struct；display_name 會跟 instrument / freq / mode / tick_size 變動。
market_profile_cols = [
    spec.display_name()
    for spec in strategy.indicators
    if spec.indicator.type == "MarketProfile"
]
daily_mp_col, ib_mp_col = market_profile_cols

mb = pl.col(daily_mp_col).struct.field("tpo_distribution")
ib = pl.col(ib_mp_col).struct.field("tpo_distribution")

df = ohlcv_result.df.filter(mb.is_not_null() | ib.is_not_null())

row = df.head(1).to_dicts()[0]
mp_data = row[daily_mp_col].pop("tpo_distribution")[0]

row


In [ ]:
pl.Config.set_fmt_str_lengths(10_000)
pl.Config.set_fmt_table_cell_list_len(100)
pl.Config.set_tbl_rows(100)

mp_df = pl.DataFrame(mp_data)
mp_df.sort("price")[::-1]


## 用 submit() 產生 Trades

`tester.submit()` 會依照極簡進出場規則產生 trades。這裡只帶到流程；產生後的 trades 會包含 MAE、MFE、PnL 等欄位，可供後續分析、篩選、分組與視覺化使用。


In [ ]:
batch = tester.submit(strategies=[strategy], periods=[period])
print(batch.status)

batch.wait(timeout=600)

summary = batch.summary()
trades = batch.all_trades()

print(summary)
print(trades.shape)
trades.tail(20)


In [ ]:
# 可選：保留結果供後續分析。Colab 會寫到 session 檔案系統。
ohlcv_df.write_parquet("simple_market_profile_ohlcv.parquet")
trades.write_parquet("simple_market_profile_trades.parquet")
print("saved parquet files")
